# Expert system and forward chaining

- **Course**: Artificial Intelligence · Master 2 (ANG, LGC, DID) · Université Yahia Farès, Médéa
- **Chapter 3**: Types of AI and approaches · Symbolic AI: expert systems
- **Duration**: about 1 h 30

### What you will do
- Read the course's rule base, which recognizes poetic forms (haiku, Italian or French sonnet, English sonnet), and describe a poem with a fact base.
- Make an inference engine reason by forward chaining, cycle by cycle, then by backward chaining.
- Work through exercise 1 of the chapter 3 tutorial (TD) to check your solution, then test the limits of an expert system.

### How to use this notebook
Run each cell from top to bottom: click the ▶ button at its left, or press Shift+Enter. **You never have to write code.** Change only the fields of the cells titled "TO CHANGE".

- In Google Colab, the menu "Runtime → Run all" runs the whole notebook; the "TO CHANGE" fields appear as check boxes, text boxes or drop-down lists.
- In Jupyter, these fields are plain `#@param` comments: change only the value after the `=` sign (`True` for a ticked box, `False` otherwise).
- After changing a value, run the cell again, then the cells that follow it.

Before each cell, **predict** the result; afterwards, compare with the "What you should see" section.

## 1. The three components of an expert system
- The **fact base**: what we know about the poem under study ("the poem has 14 lines"). It grows as the reasoning goes on.
- The **rule base**: the knowledge of the field, in the form IF premises THEN conclusion. The premises are joined by AND.
- The **inference engine**: the program that finds the applicable rules, fires them and adds their conclusions to the fact base.

Here is the course's rule base, in order of priority:

| Rule | IF (premises) | THEN (conclusion) |
|---|---|---|
| R1 | 3 lines AND 5-7-5 syllables | haiku |
| R2 | 14 lines AND 2 quatrains AND 2 tercets | sonnet |
| R3 | 14 lines AND 3 quatrains AND 1 couplet | sonnet |
| R4 | sonnet AND 2 tercets | Italian or French sonnet |
| R5 | sonnet AND 1 couplet | English sonnet |
| R6 | sonnet | fixed form |
| R7 | haiku | fixed form |

In Python, each rule is a pair (premises, conclusion). The premises form a **set**, written in curly braces: order does not matter in a set. `enumerate(rules, start=1)` numbers the rules from 1, and `" AND ".join(...)` puts the premises together with "AND".

**Before running:** how many lines will the cell display?

In [ ]:
#@title The course's rule base, in order of priority
# A rule = (set of premises, conclusion)
rules = [
    ({"3 lines", "5-7-5 syllables"}, "haiku"),                   # R1
    ({"14 lines", "2 quatrains", "2 tercets"}, "sonnet"),        # R2
    ({"14 lines", "3 quatrains", "1 couplet"}, "sonnet"),        # R3
    ({"sonnet", "2 tercets"}, "Italian or French sonnet"),       # R4
    ({"sonnet", "1 couplet"}, "English sonnet"),                 # R5
    ({"sonnet"}, "fixed form"),                                  # R6
    ({"haiku"}, "fixed form"),                                   # R7
]
for number, (premises, conclusion) in enumerate(rules, start=1):
    print(f"R{number}: IF {' AND '.join(sorted(premises))} THEN {conclusion}")

### What you should see
```
R1: IF 3 lines AND 5-7-5 syllables THEN haiku
R2: IF 14 lines AND 2 quatrains AND 2 tercets THEN sonnet
R3: IF 1 couplet AND 14 lines AND 3 quatrains THEN sonnet
R4: IF 2 tercets AND sonnet THEN Italian or French sonnet
R5: IF 1 couplet AND sonnet THEN English sonnet
R6: IF sonnet THEN fixed form
R7: IF haiku THEN fixed form
```
Seven lines, one per rule. The premises are displayed in alphabetical order (`sorted`), where digits come before letters: the order of the premises changes nothing in a rule.

**Observe:** two rules conclude "sonnet" and two conclude "fixed form". It is enough for either one to fire: rules that lead to the same fact are joined by OR.

## 2. Non-deducible facts
A **non-deducible** fact does not appear in the conclusion of any rule: only observation or the user can provide it. To find these facts, we gather all the premises, then remove all the conclusions. On sets, `|` joins and `-` removes.

**Before running:** name two non-deducible facts in this rule base.

In [ ]:
#@title Which facts are non-deducible?
all_premises = set()
all_conclusions = set()
for premises, conclusion in rules:
    all_premises = all_premises | premises          # | : join two sets
    all_conclusions.add(conclusion)
non_deducible = all_premises - all_conclusions      # - : remove
print("Non-deducible facts:", sorted(non_deducible))
print("Deducible facts:", sorted(all_conclusions))

### What you should see
```
Non-deducible facts: ['1 couplet', '14 lines', '2 quatrains', '2 tercets', '3 lines', '3 quatrains', '5-7-5 syllables']
Deducible facts: ['English sonnet', 'Italian or French sonnet', 'fixed form', 'haiku', 'sonnet']
```
The non-deducible facts are **observations**: you count lines, stanzas, syllables. "sonnet" is both a premise (of R4, R5, R6) and a conclusion (of R2, R3): it is an intermediate fact.

**Observe:** who provides the observations in a real expert system? What reading skill do you need to tick "2 tercets"?

## 3. The inference engine: forward chaining
**Forward chaining** starts from the facts. In each **cycle**, the engine goes through the rules in order of priority, fires the **first** applicable rule, adds its conclusion to the fact base, then starts again from R1: this is the course's **conflict resolution**. A rule is applicable when all its premises are known and its conclusion is new, which prevents a rule from firing twice. The engine stops when no rule produces a new fact.

`premises <= facts` means "the set of premises is included in the fact base". `break` stops going through the rules, so that the engine starts again from R1.

**Before running:** this cell defines a **function** with `def`. What will it display?

In [ ]:
#@title The inference engine with forward chaining
def forward_chaining(facts, rules):
    facts = set(facts)                      # copy: the starting base does not change
    cycle = 1
    while True:
        rule_fired = False
        for number, (premises, conclusion) in enumerate(rules, start=1):
            # applicable rule: all premises known, new conclusion
            if premises <= facts and conclusion not in facts:
                facts.add(conclusion)
                print(f"Cycle {cycle}: R{number} fired → {conclusion}")
                rule_fired = True
                break                       # conflict resolution: start again from R1
        if not rule_fired:
            print(f"Cycle {cycle}: no rule produces a new fact → stop")
            return facts
        cycle = cycle + 1

### What you should see
Nothing! A function is a recipe: `def` writes it down, but does not carry it out yet. It will be used in the next cells, under the name `forward_chaining`. This engine knows nothing about poetry: you could give it rules about medicine or grammar without changing a single line.

## 4. Describe a poem
Tick the observations that describe the poem. By default, the boxes describe Joachim Du Bellay's sonnet "Heureux qui, comme Ulysse, a fait un beau voyage" (*Les Regrets*, 1558): 14 lines, 2 quatrains, 2 tercets, in alexandrines. The cell stores these observations in a dictionary: each fact (the key) receives `True` if its box is ticked.

**Before running:** how many `True` values will the dictionary contain?

In [ ]:
#@title TO CHANGE: describe the poem (by default, Du Bellay's sonnet)
fourteen_lines = True  #@param {type:"boolean"}
three_lines = False  #@param {type:"boolean"}
two_quatrains = True  #@param {type:"boolean"}
three_quatrains = False  #@param {type:"boolean"}
two_tercets = True  #@param {type:"boolean"}
one_couplet = False  #@param {type:"boolean"}
syllables_5_7_5 = False  #@param {type:"boolean"}
alexandrines = True  #@param {type:"boolean"}

observations = {"14 lines": fourteen_lines, "3 lines": three_lines, "2 quatrains": two_quatrains,
                "3 quatrains": three_quatrains, "2 tercets": two_tercets,
                "1 couplet": one_couplet, "5-7-5 syllables": syllables_5_7_5, "alexandrines": alexandrines}
print(observations)

### What you should see
```
{'14 lines': True, '3 lines': False, '2 quatrains': True, '3 quatrains': False, '2 tercets': True, '1 couplet': False, '5-7-5 syllables': False, 'alexandrines': True}
```
Four ticked boxes, so four `True` values.

**Before running the next cell:** with the fact base {14 lines, 2 quatrains, 2 tercets, alexandrines}, which rules will fire, in which order, and in how many cycles? Work it out on paper with the table of rules.

In [ ]:
#@title Run forward chaining on this poem
base = set()
for fact, observed in observations.items():
    if observed:                                # keep only the ticked boxes
        base.add(fact)
print("Initial fact base:", sorted(base))
result = forward_chaining(base, rules)
print("Deduced facts:", sorted(result - base))

### What you should see
```
Initial fact base: ['14 lines', '2 quatrains', '2 tercets', 'alexandrines']
Cycle 1: R2 fired → sonnet
Cycle 2: R4 fired → Italian or French sonnet
Cycle 3: R6 fired → fixed form
Cycle 4: no rule produces a new fact → stop
Deduced facts: ['Italian or French sonnet', 'fixed form', 'sonnet']
```
This is the course's worked example: in cycle 1, R1 fails and R2 applies; in cycle 2, the engine starts again from R1, R2 is already done (its conclusion is known) and R4 applies; in cycle 3, R5 fails and R6 applies; in cycle 4, nothing new. Three facts have been deduced. No rule uses "alexandrines": this fact stays inactive.

**Modify:** describe Shakespeare's Sonnet 18, "Shall I compare thee to a summer's day?" (1609): untick `two_quatrains`, `two_tercets` and `alexandrines`, tick `three_quatrains` and `one_couplet`. Run both cells again. Which rules fire now?

## 5. Several poems, and a poem the system does not recognize
The cell runs the same engine on four descriptions: Du Bellay's sonnet, Shakespeare's Sonnet 18, Bashō's haiku "The old pond" (1686, 5-7-5 syllables in Japanese) and a sonnet printed as a single block, with no blank lines between the stanzas: all we observe is 14 lines.

**Before running:** will the sonnet printed as a single block be recognized as a sonnet?

In [ ]:
#@title The same engine on four poems
poems = {
    'Du Bellay, "Heureux qui, comme Ulysse..." (1558)': {"14 lines", "2 quatrains", "2 tercets", "alexandrines"},
    "Shakespeare, Sonnet 18 (1609)": {"14 lines", "3 quatrains", "1 couplet"},
    'Bashō, "The old pond" (1686)': {"3 lines", "5-7-5 syllables"},
    "A sonnet printed as a single block": {"14 lines"},
}
for title, facts in poems.items():
    print("---", title)
    result = forward_chaining(facts, rules)
    print("Deduced facts:", sorted(result - facts))

### What you should see
```
--- Du Bellay, "Heureux qui, comme Ulysse..." (1558)
Cycle 1: R2 fired → sonnet
Cycle 2: R4 fired → Italian or French sonnet
Cycle 3: R6 fired → fixed form
Cycle 4: no rule produces a new fact → stop
Deduced facts: ['Italian or French sonnet', 'fixed form', 'sonnet']
--- Shakespeare, Sonnet 18 (1609)
Cycle 1: R3 fired → sonnet
Cycle 2: R5 fired → English sonnet
Cycle 3: R6 fired → fixed form
Cycle 4: no rule produces a new fact → stop
Deduced facts: ['English sonnet', 'fixed form', 'sonnet']
--- Bashō, "The old pond" (1686)
Cycle 1: R1 fired → haiku
Cycle 2: R7 fired → fixed form
Cycle 3: no rule produces a new fact → stop
Deduced facts: ['fixed form', 'haiku']
--- A sonnet printed as a single block
Cycle 1: no rule produces a new fact → stop
Deduced facts: []
```
The English sonnet goes through R3, R5 and R6, the haiku through R1 and R7. The sonnet printed as a single block fires **no** rule: whereas a reader recognizes the form at once, the system fails as soon as you step outside the cases it was designed for. This is the **brittleness** of expert systems, one of the causes of their decline at the end of the 1980s.

**Observe:** on the other hand, each conclusion is explained by the exact list of rules that fired. What advantage does this **explainability** offer a teacher who marks a commentary on a text?

## 6. Backward chaining: starting from the goal
**Backward chaining** starts from the hypothesis to be proved. It looks for the rules that conclude it; their premises become **sub-goals**, until we get back to facts in the base. A non-deducible sub-goal that is missing from the base makes that path fail.

The `prove` function calls itself for each sub-goal, with a wider margin so that the trace reads like an outline. `all(...)` is true if all the premises are established, and stops at the first one that fails. The fact base is the one from step 4 (by default, Du Bellay).

**Before running:** can the hypothesis "English sonnet" be established for Du Bellay's sonnet? Which premise is missing?

In [ ]:
#@title TO CHANGE: test a hypothesis with backward chaining
hypothesis = "English sonnet"  #@param ["English sonnet", "Italian or French sonnet", "fixed form", "haiku"]

def prove(goal, facts, rules, margin=""):
    if goal in facts:                                  # the goal is already a known fact
        print(margin + f'"{goal}" is in the fact base: established')
        return True
    for number, (premises, conclusion) in enumerate(rules, start=1):
        if conclusion == goal:                         # a rule that concludes the goal
            print(margin + f'To establish "{goal}", R{number} requires: {sorted(premises)}')
            if all(prove(p, facts, rules, margin + "    ") for p in sorted(premises)):
                return True                            # all the sub-goals are established
    print(margin + f'"{goal}" cannot be established')
    return False

print("Hypothesis established:", prove(hypothesis, base, rules))

### What you should see
```
To establish "English sonnet", R5 requires: ['1 couplet', 'sonnet']
    "1 couplet" cannot be established
"English sonnet" cannot be established
Hypothesis established: False
```
By R5, "English sonnet" requires "1 couplet" and "sonnet". "1 couplet" is non-deducible and missing from the base: the hypothesis is rejected at once, without even examining "sonnet".

**Modify:** choose `fixed form` in the list. The trace goes down from "fixed form" to "sonnet" (R6), then from "sonnet" to the three observations (R2). Compare with the forward chaining of step 4: which of the two examines fewer rules? Which one suits diagnosis, when you have a hypothesis to check?

## 7. Chapter 3 tutorial (TD), exercise 1
Consider the following rules in forward chaining. The symbol ∧ means "AND" (the syllabus writes it ∩): "E ∧ G → C" reads "if E and G are established, then C is established".

| Rule | Conditions | Conclusions |
|---|---|---|
| R1 | A | E |
| R2 | B | D |
| R3 | H | A ∧ F |
| R4 | E ∧ G | C |
| R5 | E ∧ K | B |
| R6 | D ∧ E ∧ K | C |
| R7 | G ∧ K ∧ F | A |

- **Rule order**: R3 R7 R1 R4 R5 R2 R6
- **Non-deducible facts**: H, G, K
- **Initial fact base**: H, K
- **Goal**: C

Questions: a) draw the graph that corresponds to the rules; b) work through the system in forward chaining. Try on paper first, then check with the next cells.

Here a rule can have several conclusions (R3 concludes A and F): each rule is stored in a dictionary, under its name, with the set of its conditions and the set of its conclusions.

**Before running:** which rules conclude A? Which ones conclude C?

In [ ]:
#@title The rules of tutorial exercise 1
# rule name: (set of conditions, set of conclusions)
rules_td = {
    "R1": ({"A"}, {"E"}),
    "R2": ({"B"}, {"D"}),
    "R3": ({"H"}, {"A", "F"}),
    "R4": ({"E", "G"}, {"C"}),
    "R5": ({"E", "K"}, {"B"}),
    "R6": ({"D", "E", "K"}, {"C"}),
    "R7": ({"G", "K", "F"}, {"A"}),
}
for name, (conditions, conclusions) in rules_td.items():
    print(f"{name}: {' ∧ '.join(sorted(conditions))} → {' ∧ '.join(sorted(conclusions))}")

### What you should see
```
R1: A → E
R2: B → D
R3: H → A ∧ F
R4: E ∧ G → C
R5: E ∧ K → B
R6: D ∧ E ∧ K → C
R7: F ∧ G ∧ K → A
```
The conditions are displayed in alphabetical order: R7 is written here "F ∧ G ∧ K", which amounts to the same thing as "G ∧ K ∧ F". A is concluded by R3 and R7, C by R4 and R6.

### a) The rule graph
In the AND/OR graph, each fact is a rectangle and each rule a circle. An arrow goes from each condition to the rule, then from the rule to each of its conclusions. The `networkx` library builds this directed graph; `in_degree(n)` counts the arrows that point to `n`.

**Before running:** how many arrows does the graph have? Which facts have no arrow pointing to them?

In [ ]:
#@title a) Build the AND/OR graph of the rules
import networkx as nx

graph = nx.DiGraph()                         # a directed graph: nodes and arrows
for name, (conditions, conclusions) in rules_td.items():
    for fact in conditions:
        graph.add_edge(fact, name)           # arrow: condition → rule
    for fact in conclusions:
        graph.add_edge(name, fact)           # arrow: rule → conclusion
print(graph.number_of_nodes(), "nodes and", graph.number_of_edges(), "arrows")
no_incoming_arrow = [n for n in graph if graph.in_degree(n) == 0]
print("Facts with no arrow pointing to them:", sorted(no_incoming_arrow))

### What you should see
```
16 nodes and 21 arrows
Facts with no arrow pointing to them: ['G', 'H', 'K']
```
16 nodes (9 facts and 7 rules) and 21 arrows: 13 conditions and 8 conclusions. No arrow points to G, H or K: these are indeed the non-deducible facts of the exercise.

**Before running the next cell:** on your scrap paper, trace the path that leads from H and K to C. Does it go through R4?

In [ ]:
#@title Draw the graph: rectangles = facts, circles = rules
import matplotlib.pyplot as plt

position = {"H": (2, 0), "R3": (2, 1), "F": (1, 1.5), "G": (0, 2.5), "K": (0, 4.5),
            "R7": (1, 3), "A": (2, 3), "R1": (3, 3), "E": (4, 3), "R4": (5, 1.5),
            "R5": (5, 4.5), "B": (6, 4.5), "R2": (7, 4.5), "D": (8, 4.5), "R6": (9, 3), "C": (10, 3)}
the_rules = [n for n in graph if n.startswith("R")]
the_facts = [n for n in graph if not n.startswith("R")]
plt.figure(figsize=(10, 5))
nx.draw_networkx_nodes(graph, position, nodelist=the_facts, node_shape="s", node_size=800, node_color="#dbe9f6", edgecolors="black")
nx.draw_networkx_nodes(graph, position, nodelist=the_rules, node_shape="o", node_size=800, node_color="#fde2c4", edgecolors="black")
nx.draw_networkx_labels(graph, position)
nx.draw_networkx_edges(graph, position, node_size=800, arrowsize=15)
plt.title("AND/OR graph of rules R1 to R7 (chapter 3 tutorial, exercise 1)")
plt.axis("off")
plt.show()

### What you should see
A graph with 16 nodes: H at the bottom, below R3; K and G on the left; C on the right, at the end of the chain R1 → E → R5 → B → R2 → D → R6. The arrows that enter the same rule are joined by AND; when two rules lead to the same fact (R3 and R7 for A, R4 and R6 for C), they are joined by OR.

**Observe:** G is neither in the initial base nor deducible. So rules R7 and R4, which need it, can never fire, and the only possible path to C goes through R6. This is the diagnosis given in the solution.

### b) Work through forward chaining
The tutorial solution examines the rules in the required order. A rule fires if all its conditions are in the fact base; its conclusions are then added to the base. A rule that has already fired is not run again. We stop as soon as the goal is in the base, or when a complete **pass** over the rules adds nothing new.

The "TO CHANGE" cell prepares the rule order, the initial fact base and the goal. Type the names separated by spaces.

**Before running:** what will the last two lines display?

In [ ]:
#@title TO CHANGE: rule order, initial fact base and goal
order = "R3 R7 R1 R4 R5 R2 R6"  #@param {type:"string"}
initial_base = "H K"  #@param {type:"string"}
goal = "C"  #@param ["C", "D", "B", "E", "A"]

ordered_rules = []
for name in order.upper().split():          # "R3 R7 ..." becomes ["R3", "R7", ...]
    if name in rules_td:                    # unknown names are ignored
        ordered_rules.append((name, rules_td[name]))
initial_facts = set(initial_base.upper().split())
print("Order:", [name for name, rule in ordered_rules])
print("Initial fact base:", sorted(initial_facts), "| goal:", goal)

### What you should see
```
Order: ['R3', 'R7', 'R1', 'R4', 'R5', 'R2', 'R6']
Initial fact base: ['H', 'K'] | goal: C
```
**Before running the next cell:** in which order will the rules fire? Will the goal C be reached, and in how many passes?

The next cell follows the program from the tutorial solution and numbers the passes. There is at most one pass per rule, since each useful pass fires at least one new rule.

In [ ]:
#@title b) The tutorial's forward chaining, pass by pass
facts = set(initial_facts)                    # copy of the starting base
fired = []
for pass_number in range(1, len(ordered_rules) + 1):   # one pass = one full round of the rules
    new_fact = False
    for name, (conditions, conclusions) in ordered_rules:
        # a rule fires only once, if its conditions are in the base (<=)
        if name not in fired and conditions <= facts and goal not in facts:
            facts = facts | conclusions       # add the conclusions
            fired.append(name)
            new_fact = True
            print(f"Pass {pass_number} · {name} fired -> {sorted(facts)}")
    if not new_fact or goal in facts:         # nothing new, or goal reached: stop
        break
print("Goal reached:", goal in facts, "| rules fired:", fired)

### What you should see
```
Pass 1 · R3 fired -> ['A', 'F', 'H', 'K']
Pass 1 · R1 fired -> ['A', 'E', 'F', 'H', 'K']
Pass 1 · R5 fired -> ['A', 'B', 'E', 'F', 'H', 'K']
Pass 1 · R2 fired -> ['A', 'B', 'D', 'E', 'F', 'H', 'K']
Pass 1 · R6 fired -> ['A', 'B', 'C', 'D', 'E', 'F', 'H', 'K']
Goal reached: True | rules fired: ['R3', 'R1', 'R5', 'R2', 'R6']
```
The goal C is established in the first pass. The final fact base is {H, K, A, F, E, B, D, C} and the rules that fired are, in order, **R3, R1, R5, R2, R6**. R7 and R4 never fire, since G is neither in the initial base nor deducible.

**Modify**, in the "TO CHANGE" cell, then run both cells again:
1. `initial_base` = `H G K`: R7 and R4 can now fire, and the goal is reached earlier, through R4. Which rules fire?
2. `order` = `R6 R2 R5 R4 R1 R7 R3` (the reverse order): do the same rules fire? How many passes are needed now?
3. `initial_base` = `K`: why does nothing fire?

### The solution table, step by step
The solution presents the first pass as a table: each rule examined, its condition, whether it fires and the fact base after the examination. The `pandas` library stores these rows in a table (a `DataFrame`).

**Before running:** how many rows does the table have? What does the "Fires?" column say for R7 and for R4?

In [ ]:
#@title The table of the first pass, as in the solution
import pandas as pd

facts = set(initial_facts)
rows = []
for step, (name, (conditions, conclusions)) in enumerate(ordered_rules, start=1):
    if conditions <= facts:
        facts = facts | conclusions
        answer = "Yes"
    else:
        answer = "No: " + ", ".join(sorted(conditions - facts)) + " missing"
    rows.append([step, name, " ∧ ".join(sorted(conditions)), answer, ", ".join(sorted(facts))])
    if goal in facts:
        break
pd.DataFrame(rows, columns=["Step", "Rule examined", "Condition", "Fires?", "Fact base afterwards"])

### What you should see
| | Step | Rule examined | Condition | Fires? | Fact base afterwards |
|---|---|---|---|---|---|
| 0 | 1 | R3 | H | Yes | A, F, H, K |
| 1 | 2 | R7 | F ∧ G ∧ K | No: G missing | A, F, H, K |
| 2 | 3 | R1 | A | Yes | A, E, F, H, K |
| 3 | 4 | R4 | E ∧ G | No: G missing | A, E, F, H, K |
| 4 | 5 | R5 | E ∧ K | Yes | A, B, E, F, H, K |
| 5 | 6 | R2 | B | Yes | A, B, D, E, F, H, K |
| 6 | 7 | R6 | D ∧ E ∧ K | Yes | A, B, C, D, E, F, H, K |

The unlabeled column on the left is the table's index (0 to 6).

This is the solution table, except for the order in which the facts are written: here, the base is sorted alphabetically. Compare it with yours, row by row.

**Observe:** with `initial_base` = `H G K`, the table stops at step 4, on R4. Has the engine "understood" anything? Justify your answer: does it reason, or does it simply check whether one set is included in another?

## Your turn
1. In step 4, describe Shakespeare's Sonnet 18 (3 quatrains and 1 couplet), then a haiku (3 lines, 5-7-5 syllables): check that the rules that fire are the ones you predicted.
2. In the tutorial exercise, add G to the initial base, then change the rule order: note what changes and what does not.
3. The system does not know the villanelle, a fixed form of 19 lines: 5 tercets and 1 quatrain, for example Oscar Wilde, "Theocritus: A Villanelle" (1881). The cell below adds a rule R8 written in the fields, then runs the engine on the poem you describe. Separate the facts with commas.

In [ ]:
#@title TO CHANGE: add your own rule, then describe a poem
if_part = "19 lines, 5 tercets, 1 quatrain"  #@param {type:"string"}
then_part = "villanelle"  #@param {type:"string"}
facts_of_the_poem = "19 lines, 5 tercets, 1 quatrain"  #@param {type:"string"}

premises = set()
for piece in if_part.split(","):           # "19 lines, 5 tercets" → {"19 lines", "5 tercets"}
    premises.add(piece.strip())            # strip() removes extra spaces
facts = set()
for piece in facts_of_the_poem.split(","):
    facts.add(piece.strip())
extended_rules = rules + [(premises, then_part.strip())]   # R8: the new rule, added last
result = forward_chaining(facts, extended_rules)
print("Deduced facts:", sorted(result - facts))

### What you should see
```
Cycle 1: R8 fired → villanelle
Cycle 2: no rule produces a new fact → stop
Deduced facts: ['villanelle']
```
**Observe:** "fixed form" has not been deduced, although the villanelle is one. What would you have to add to the rule base? Each new form requires several rules, written by hand: this is the **knowledge acquisition bottleneck**. Then try `facts_of_the_poem` = `19 lines, 5 tercets, 1 quatrain, sonnet`: what happens, and why does the system not see the contradiction?

## For the mini-project: your own rule base
The engine does not depend on the field: you can write your rules in a spreadsheet, one rule per row, with three columns `rule`, `if` (the premises, separated by commas) and `then` (the conclusion). Save it as "CSV UTF-8" with **semicolons** as separators, since the premises contain commas. The example rules, which recognize the genre of a text, are **simplified and made up**: discuss them before using them.

- In Colab, tick `load_my_file`: a window opens so that you can choose the file.
- In Jupyter, put the file next to the notebook and type its name in `file_name`.

**Before running:** how many rules and columns does the example table have?

In [ ]:
#@title A rule base in a table (made-up example: the genre of a text)
import io

example_rules = """rule;if;then
R1;dialogue, stage directions;play
R2;lines of verse, rhymes;verse poem
R3;narrator, chapters;narrative
R4;narrative, long;novel
R5;narrative, short;short story"""
rules_table = pd.read_csv(io.StringIO(example_rules), sep=";")   # sep: the separator
print(rules_table.shape)
rules_table

### What you should see
`(5, 3)`, then a table of 5 rules and 3 columns: `rule`, `if`, `then`. R3 and R4 are linked: the conclusion of R3 ("narrative") is a premise of R4 and R5.

In [ ]:
#@title TO CHANGE: load your own rule base (optional)
load_my_file = False  #@param {type:"boolean"}
file_name = "my_rules.csv"  #@param {type:"string"}

if load_my_file:
    try:
        from google.colab import files        # this module only exists in Colab
        upload = files.upload()               # a window opens: choose your file
        file_name = list(upload)[0]
    except ImportError:
        print("Not in Colab: the file", file_name, "must be next to the notebook.")
    rules_table = pd.read_csv(file_name, sep=";", encoding="utf-8")
print(len(rules_table), "rules")
print(rules_table.head())

### What you should see
As long as the box is not ticked, the 5 example rules. With your own file, check that the three columns are called `rule`, `if` and `then`.

**Before running the next cell:** for a text that has a narrator, chapters and is long, which rules will fire?

In [ ]:
#@title TO CHANGE: make the engine reason with your rule base
facts_of_the_text = "narrator, chapters, long"  #@param {type:"string"}

table_rules = []
for i, row in rules_table.iterrows():         # one row of the table = one rule
    premises = set()
    for piece in row["if"].split(","):
        premises.add(piece.strip())
    table_rules.append((premises, row["then"].strip()))
facts = set()
for piece in facts_of_the_text.split(","):
    facts.add(piece.strip())
result = forward_chaining(facts, table_rules)   # R1 = first row of the table
print("Deduced facts:", sorted(result - facts))

### What you should see
```
Cycle 1: R3 fired → narrative
Cycle 2: R4 fired → novel
Cycle 3: no rule produces a new fact → stop
Deduced facts: ['narrative', 'novel']
```
R3 concludes "narrative", which allows R4 to conclude "novel". The same engine, without any change, now reasons about genres.

**For your mini-project:** write about ten rules from your own field (recognizing a register, a figure of speech, the level of a text for learners…), test them on cases you know, then look for a borderline case that makes them fail.

## To hand in
Answer in writing, in English, in a few lines each:
1. Explain in your own words the difference between forward and backward chaining, using the two traces you obtained for Du Bellay's sonnet (steps 4 and 6).
2. The sonnet printed as a single block fires no rule, whereas a reader recognizes it at once. What does this failure reveal about the strengths and limits of expert systems?
3. Copy out the process you obtained for tutorial exercise 1 with the base {H, K}, then say what changes when G is added to the initial base. Why is the goal reached earlier?

## Key points
- An expert system brings together a fact base, a base of IF … THEN rules and an inference engine; the engine does not depend on the field.
- Forward chaining goes from the facts to the goal, cycle after cycle, following the priority order of the rules; backward chaining goes from the goal to the facts, through sub-goals.
- Its strength is explainability: each conclusion is justified by the chain of rules that fired. Its limits: all the rules must be written by hand, and it fails outside the cases it was designed for.

**In the course:** [Symbolic AI: expert systems](https://progremer04.github.io/canva-au-for-ang-/en/#c3-symbolique) · [Chapter 3 tutorial, exercise 1](https://progremer04.github.io/canva-au-for-ang-/en/#td-ch3-ex1)